# Named Entity Recognition

a great example:
https://www.kaggle.com/code/bennyfung/sentiment-and-ner-analysis-of-audit-comments#7.-Named-Entity-Recognition-(NER) 

a paper that is relevant:
https://arxiv.org/abs/2302.11157

another relevant paper:
https://dl.acm.org/doi/10.1145/3649451#sec-4 

In [6]:
# imports
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import ast

In [20]:
stocks_df = pd.read_csv('../data/ASXListedCompanies.csv')
stocks_df['asx_code'] = 'ASX:' + stocks_df['ASX code'].astype(str)
stock_dict = dict(zip(stocks_df['asx_code'], stocks_df['Company name'].str.lower()))


In [11]:
articles_df = pd.read_csv('../data/article_scraped_data.csv', converters={'stock_codes': ast.literal_eval}).dropna(subset=['document'])
articles_df = articles_df[articles_df['stock_codes'].apply(lambda x: len(x) > 0)]
articles_df.dropna(subset=['document'])
articles_df

,title,author,date_string,sector,stock_codes,document,url
3,GreenHy2 secures H2Core deal to advance superc...,Colin Hay,"March 21, 2025",Energy,[ASX:H2G],Solid state hydrogen storage developer GreenHy...,https://smallcaps.com.au/greenhy2-h2core-deal-...
4,Compumedics reaches $20m in Chinese MEG sales ...,Colin Hay,"March 21, 2025",Biotech,[ASX:CMP],Brain research technologies specialist Compume...,https://smallcaps.com.au/compumedics-chinese-m...
5,CZR Resources receives $75m offer for Robe Mes...,Colin Hay,"March 21, 2025",Mining,[ASX:CZR],CZR Resources (ASX: CZR) has received a condit...,https://smallcaps.com.au/czr-resources-offer-r...
6,Terbium tipped to follow gallium’s boom as dem...,Colin Hay,"March 21, 2025",Hot Topics,"[ASX:ASM, ASX:NTU, ASX:VTM]",Gallium has become the darling of markets and ...,https://smallcaps.com.au/terbium-follow-galliu...
7,Arafura signs five-year offtake deal with Trax...,Colin Hay,"March 20, 2025",Mining,[ASX:ARU],Arafura Rare Earths (ASX: ARU) has signed a bi...,https://smallcaps.com.au/arafura-five-year-off...
...,...,...,...,...,...,...,...
1701,Adriatic Metals produces first concentrate at ...,Colin Hay,"February 29, 2024",Mining,[ASX:ADT],Adriatic Metals (ASX: ADT) has overcome a deli...,https://smallcaps.com.au/adriatic-metals-produ...
1702,Nova Minerals targets NASDAQ dual listing to i...,Colin Hay,"February 29, 2024",Mining,[ASX:NVA],Nova Minerals (ASX: NVA) has confirmed its pro...,https://smallcaps.com.au/nova-minerals-targets...
1703,Radiopharm Theranostics doses first patient in...,Imelda Cotton,"February 29, 2024",Biotech,[ASX:RAD],Biotechnology company Radiopharm Theranostics ...,https://smallcaps.com.au/radiopharm-theranosti...
1704,Boss Energy on track to produce first drum of ...,Imelda Cotton,"February 29, 2024",Mining,[ASX:BOE],Boss Energy (ASX: BOE) has started commissioni...,https://smallcaps.com.au/boss-energy-on-track-...


In [25]:
import spacy
from spacy.pipeline import EntityRuler

# Invert the dictionary for matching
# "bhp group limited" → "ASX:BHP", "bhp" → "ASX:BHP"
asx_map = {
    value.lower(): key  # full name
    for key, value in stock_dict.items()
}

# Add ASX code without prefix too (like "bhp")
for key in stock_dict:
    code = key.replace("ASX:", "").lower()
    asx_map[code] = key

# Create a spaCy blank pipeline and add the EntityRuler
nlp = spacy.blank("en")
ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})

# Create patterns for both company names and ASX codes
patterns = [{"label": "ORG", "pattern": name} for name in asx_map.keys()]
ruler.add_patterns(patterns)

def extract_stocks_spacy(text):
    doc = nlp(text)
    extracted_stocks = set()
    
    for ent in doc.ents:
        if ent.label_ == "ORG":
            name = ent.text.lower()
            if name in asx_map:
                extracted_stocks.add(asx_map[name])
    
    return list(extracted_stocks)
sample_text = "Wesfarmers and asx:CSL are performing well. asx:bhp and 1414 Degrees are down slightly."
print("Extracted Stocks (spaCy):", extract_stocks_spacy(sample_text))
print([ent.text for ent in nlp(sample_text).ents])


Extracted Stocks (spaCy): ['ASX:ASX', 'ASX:BHP']
['asx', 'asx', 'bhp']


In [5]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

# Load NER model
ner_model_name = "dslim/bert-base-NER"  # Consider finance-specific NER models if needed
ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_name)
ner_model = AutoModelForTokenClassification.from_pretrained(ner_model_name)


# Create NER pipeline
ner_pipeline = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer)
def extract_stocks(text):
    """Extract stock mentions using a Transformer-based NER model."""
    ner_results = ner_pipeline(text)
    
    extracted_stocks = set()
    
    for entity in ner_results:
        if entity["entity"].endswith("ORG"):  # Organizations (Companies)
            stock_name = entity["word"].replace("##", "").strip()
            
            # Match with ASX codes
            if stock_name.lower() in stock_dict:
                extracted_stocks.add(stock_dict[stock_name.lower()])
    
    return list(extracted_stocks)

# Test on a sample article
sample_text = articles_df.iloc[0]["document"]
print("Extracted Stocks:", extract_stocks(sample_text))

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Extracted Stocks: []


In [5]:
sample_text = "BHP announced record profits this quarter. Investors are bullish."
print("Extracted Stocks:", extract_stocks(sample_text))

Extracted Stocks: []


In [ ]:
from transformers import BertTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf

# Load sentiment analysis model (FinBERT is trained for finance sentiment tasks)
sentiment_model_name = "ProsusAI/finbert"
sentiment_tokenizer = BertTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = TFAutoModelForSequenceClassification.from_pretrained(sentiment_model_name)

def classify_sentiment(text):
    """Predict sentiment for a given text using FinBERT."""
    inputs = sentiment_tokenizer(text, return_tensors="tf", padding=True, truncation=True, max_length=512)
    outputs = sentiment_model(**inputs)
    scores = tf.nn.softmax(outputs.logits, axis=1).numpy()[0]

    # Map index to sentiment labels
    sentiment_labels = {0: "Negative", 1: "Neutral", 2: "Positive"}
    sentiment = sentiment_labels[np.argmax(scores)]
    
    return sentiment

# Test sentiment function
sample_text = "BHP announced record profits this quarter. Investors are bullish."
print("Sentiment:", classify_sentiment(sample_text))
